In [1]:
!pwd

/DATA/lepetit/ls-test/Pipeline/1-TimeGrad


In [3]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

from metrics2 import DatasetEvaluator


# ==============================================================================
# 1. ORIENTATION DES MÉTRIQUES (Sens du tri pour le ranking : True = plus petit = meilleur)
# ==============================================================================
LOWER_IS_BETTER_CONT = {
    "MSE": True, "MAE": True, "MAPE": True, "SMAPE": True,
    "JS": True, "KL": True, "MMD": True, "R2": False
}

LOWER_IS_BETTER_CAT = {
    "accuracy_globale": False, "accuracy_minoritaire": False,
    "macro_f1_arith": False, "weighted_f1": False,
    "fsg_macro": False, "fsg_weighted": False,
    "gmean_strict": False, "gmean_smoothed": False,
    "macro_auc_roc": False, "weighted_auc_roc": False,
    "auc_pr_macro": False
}

COL_NAMES = ["FC", "PAS", "PAM", "PAD", "Temp", "SpO2", "FR", "event_code"]


# ==============================================================================
# 2. PARCOURS ET ÉVALUATION
# ==============================================================================
def collect_and_evaluate(root_dir: str):
    """
    Parcourt récursivement root_dir à la recherche des dossiers contenant
    real_data.npy et gen_data.npy, puis calcule les métriques.
    """
    root_path = Path(root_dir).resolve()
    if not root_path.exists():
        raise FileNotFoundError(f"Chemin racine introuvable : {root_path}")

    cont_records = []
    cat_records = []

    print(f"Recherche récursive dans : {root_path} ...\n")

    for dirpath, _, filenames in os.walk(root_path):
        if "real_data.npy" in filenames and "gen_data.npy" in filenames:
            folder = Path(dirpath)
            config_name = str(folder.relative_to(root_path))
            print(f"-> Traitement de : {config_name}")

            real_data = np.load(folder / "real_data.npy")
            gen_data = np.load(folder / "gen_data.npy")

            # Nom de dossier sûr pour Windows / POSIX
            safe_name = config_name.replace(os.sep, "_").replace("/", "_").replace("\\", "_")

            try:
                evaluator = DatasetEvaluator(
                    real_data,
                    gen_data,
                    col_names=COL_NAMES,
                    path_dir=f"checkpoints/{safe_name}",
                )
                res_cont, res_cat = evaluator.run_full_analysis()
            except Exception as e:
                print(f"   [ERREUR] Évaluation impossible pour {config_name} : {e}")
                continue

            # --- Variables continues ---
            if isinstance(res_cont, dict):
                records_list = res_cont.get("records", [])
            else:
                records_list = res_cont

            for r in records_list:
                if not isinstance(r, dict):
                    continue
                item = r.copy()
                item["config"] = config_name
                cont_records.append(item)

            # --- Variable catégorielle ---
            if isinstance(res_cat, dict):
                item = res_cat.copy()
                item["config"] = config_name
                cat_records.append(item)

    df_cont = pd.DataFrame(cont_records)
    df_cat = pd.DataFrame(cat_records)
    return df_cont, df_cat


# ==============================================================================
# 3. CALCUL DES RANKINGS DÉTAILLÉS ET GLOBAUX
# ==============================================================================
def _mean_or_nan(df: pd.DataFrame, cols: list) -> pd.Series:
    """Moyenne ligne par ligne en tolérant une liste de colonnes vide."""
    if not cols:
        return pd.Series(np.nan, index=df.index)
    return df[cols].mean(axis=1)


def compute_all_rankings(df_cont: pd.DataFrame, df_cat: pd.DataFrame):
    """
    Calcule :
    - Le rang par variable et par métrique
    - Le leaderboard par variable (meilleure config pour chaque variable)
    - Le leaderboard catégoriel
    - Le leaderboard global consolidé
    """
    # --------------------------------------------------------------------------
    # A. Ranking Variables Continues
    # --------------------------------------------------------------------------
    df_cont_ranked = df_cont.copy()
    cont_metrics = [c for c in df_cont.columns if c not in ["config", "variable"]]

    for m in cont_metrics:
        asc = LOWER_IS_BETTER_CONT.get(m, True)
        df_cont_ranked[f"{m}_rank"] = (
            df_cont_ranked.groupby("variable")[m]
            .rank(ascending=asc, method="min")
        )

    pointwise_cols = [f"{m}_rank" for m in ["MSE", "MAE", "MAPE", "SMAPE", "R2"]
                      if f"{m}_rank" in df_cont_ranked]
    distrib_cols   = [f"{m}_rank" for m in ["JS", "KL", "MMD"]
                      if f"{m}_rank" in df_cont_ranked]

    df_cont_ranked["rank_pointwise"]    = _mean_or_nan(df_cont_ranked, pointwise_cols)
    df_cont_ranked["rank_distribution"] = _mean_or_nan(df_cont_ranked, distrib_cols)
    df_cont_ranked["var_score"] = (
        df_cont_ranked["rank_pointwise"] + df_cont_ranked["rank_distribution"]
    ) / 2

    # Pivot robuste (tolère les doublons de (config, variable))
    best_per_var_pivot = df_cont_ranked.pivot_table(
        index="config", columns="variable", values="var_score", aggfunc="mean"
    )

    best_config_summary = []
    for var in best_per_var_pivot.columns:
        col = best_per_var_pivot[var].dropna()
        if col.empty:
            continue
        best_cfg = col.idxmin()
        best_rank = col.loc[best_cfg]
        best_config_summary.append({
            "Variable": var,
            "Meilleure_Configuration": best_cfg,
            "Rang_Moyen": round(best_rank, 2),
        })
    df_best_by_var = pd.DataFrame(best_config_summary)

    cont_global_score = (
        df_cont_ranked.groupby("config")["var_score"]
        .mean()
        .reset_index()
        .rename(columns={"var_score": "score_continuous"})
    )

    # --------------------------------------------------------------------------
    # B. Ranking Variable Catégorielle
    # --------------------------------------------------------------------------
    df_cat_ranked = df_cat.copy()
    cat_metrics = [c for c in df_cat.columns if c != "config"]

    for m in cat_metrics:
        asc = LOWER_IS_BETTER_CAT.get(m, False)
        df_cat_ranked[f"{m}_rank"] = df_cat_ranked[m].rank(ascending=asc, method="min")

    imbalanced_cols = [f"{m}_rank" for m in
                       ["accuracy_minoritaire", "macro_f1_arith", "fsg_macro",
                        "gmean_smoothed", "auc_pr_macro"]
                       if f"{m}_rank" in df_cat_ranked]
    global_cols = [f"{m}_rank" for m in
                   ["accuracy_globale", "weighted_f1", "fsg_weighted", "weighted_auc_roc"]
                   if f"{m}_rank" in df_cat_ranked]

    df_cat_ranked["rank_imbalanced"]  = _mean_or_nan(df_cat_ranked, imbalanced_cols)
    df_cat_ranked["rank_global_cat"]  = _mean_or_nan(df_cat_ranked, global_cols)
    df_cat_ranked["score_categorical"] = (
        df_cat_ranked["rank_imbalanced"] + df_cat_ranked["rank_global_cat"]
    ) / 2

    cat_global_score = df_cat_ranked[["config", "score_categorical"]].copy()

    # --------------------------------------------------------------------------
    # C. Leaderboard Global Consolidé
    # --------------------------------------------------------------------------
    leaderboard = pd.merge(cont_global_score, cat_global_score, on="config", how="outer")
    leaderboard["global_rank_score"] = (
        leaderboard[["score_continuous", "score_categorical"]].mean(axis=1)
    )
    leaderboard["Rang_Final"] = (
        leaderboard["global_rank_score"].rank(ascending=True, method="min").astype("Int64")
    )
    leaderboard = leaderboard.sort_values("Rang_Final", na_position="last").reset_index(drop=True)

    return df_cont_ranked, best_per_var_pivot, df_best_by_var, df_cat_ranked, leaderboard


# ==============================================================================
# 4. EXPORT ET VISUALISATION
# ==============================================================================
def save_results(df_cont_ranked,
                 best_per_var_pivot,
                 df_best_by_var,
                 df_cat_ranked,
                 leaderboard,
                 output_dir="results_benchmark"):
    """
    Exporte l'ensemble des résultats et rankings sous forme de fichiers CSV.
    """
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    leaderboard.to_csv(out_path / "leaderboard_global.csv", index=False)
    df_best_by_var.to_csv(out_path / "meilleure_par_variable.csv", index=False)
    best_per_var_pivot.to_csv(out_path / "rangs_variables_pivot.csv", index=True)
    df_cont_ranked.to_csv(out_path / "details_continues.csv", index=False)
    df_cat_ranked.to_csv(out_path / "details_categorielle.csv", index=False)

    print(f"\n Tous les fichiers CSV ont été enregistrés dans : {out_path.resolve()}")
    print("\n" + "=" * 80)
    print("TOP 5 DES MEILLEURES CONFIGURATIONS GLOBALES :")
    print("=" * 80)
    print(
        leaderboard[["Rang_Final", "config", "score_continuous",
                     "score_categorical", "global_rank_score"]]
        .head(5).to_string(index=False)
    )
    print("\n" + "=" * 80)
    print("MEILLEURE CONFIGURATION PAR VARIABLE :")
    print("=" * 80)
    print(df_best_by_var.to_string(index=False))


# ==============================================================================
# 5. POINT D'ENTRÉE
# ==============================================================================
def main():
    # 1. Parcours récursif et calcul
    df_cont, df_cat = collect_and_evaluate("../1-TimeGrad")

    if df_cont.empty or df_cat.empty:
        raise RuntimeError("Aucune donnée trouvée : vérifiez le chemin racine ../results")

    # 2. Calcul des rankings
    (
        df_cont_ranked,
        best_per_var_pivot,
        df_best_by_var,
        df_cat_ranked,
        leaderboard,
    ) = compute_all_rankings(df_cont, df_cat)

    # 3. Export
    save_results(
        df_cont_ranked,
        best_per_var_pivot,
        df_best_by_var,
        df_cat_ranked,
        leaderboard,
        output_dir="resultats_benchmark",
    )


if __name__ == "__main__":
    main()

Recherche récursive dans : /DATA/lepetit/ls-test/Pipeline/1-TimeGrad ...

-> Traitement de : checkpoints_pred16_hist32_nl3_nc80_rc32_bs128_cce0.25
                             RAPPORT D'ÉVALUATION                                   

=== ÉVALUATION DES VARIABLES CONTINUES ===

---------------------------------------------------------------------------------------------------
Variable   | MSE       | MAE       | MAPE (%)  | SMAPE (%)  | R²      | JS      | KL      | MMD    
---------------------------------------------------------------------------------------------------
FC         | 113.7483  | 4.8372    | 4.89      | 5.17       | 0.876   | 0.0820  | 0.0974  | 0.0009 
PAS        | 131.0809  | 5.3947    | 4.98      | 4.90       | 0.887   | 0.0753  | 0.0368  | 0.0007 
PAM        | 47.3347   | 2.7945    | 3.85      | 3.89       | 0.889   | 0.0587  | 0.0141  | 0.0006 
PAD        | 24.2933   | 2.0542    | 3.77      | 3.88       | 0.901   | 0.0896  | 0.0323  | 0.0013 
Temp       | 3.7188    